In [0]:
# ==============================================================================
# 1. CONFIGURATION & PATHS
# ==============================================================================
spark.sql("CREATE DATABASE IF NOT EXISTS olist_db")
orders_path = "/Workspace//tables/olist_orders_dataset.csv"
customers_path = "/Workspace//tables/olist_customers_dataset.csv"

# =================================================================
# 2. READ DATA WITH SCHEMA INFERENCING
# =================================================================
orders_df = spark.read.csv(orders_path, header=True, inferSchema=True)
customers_df = spark.read.csv(customers_path, header=True, inferSchema=True)

print(f"Orders count: {orders_df.count()}")
print(f"Customers count: {customers_df.count()}")

# =================================================================
# 3. DATA CLEANING & TRANSFORMATION (The "Engineering" Part)
# =================================================================
from pyspark.sql.functions import col, to_date, datediff, current_date, when

clean_orders = orders_df.dropna(subset=["order_id", "customer_id", "order_purchase_timestamp"])

clean_customers = customers_df.dropDuplicates(["customer_unique_id"])

clean_orders = clean_orders.withColumn("purchase_date", to_date(col("order_purchase_timestamp"))) \
                           .withColumn("delivery_date", to_date(col("order_delivered_customer_date")))

clean_orders = clean_orders.withColumn(
    "delivery_time_days",
    when(col("delivery_date").isNull(), -1)  # -1 indicates not yet delivered
    .otherwise(datediff(col("delivery_date"), col("purchase_date")))
)

# ================================================================
# 4. DISTRIBUTED JOIN
# ================================================================
analytics_df = clean_orders.join(
    clean_customers.select("customer_id", "customer_unique_id", "customer_state", "customer_city"),
    on="customer_id",
    how="left"
)

# ===============================================================
# 4.5 DATA QUALITY ASSERTIONS (Native PySpark)
# ===============================================================
print("Running Data Quality Checks...")

assert analytics_df.filter(col("customer_id").isNull()).count() == 0, "FAIL:Nulls found in customer_id column"
assert analytics_df.filter(col("order_id").isNull()).count() == 0, "FAIL:Nulls found in order_id column"

valid_deliveries = analytics_df.filter(col("delivery_time_days") != -1)
assert valid_deliveries.filter(col("delivery_time_days") < 0).count() == 0, "FAIL:Negative delivery_time_days found"

original_count = clean_orders.count()
final_count = analytics_df.count()
assert final_count >= original_count *0.95, f"Signficant data loss! Expected at least 95% of original data, found {final_count} out of {original_count} records"

print("All Data Quality Checks Passed!")

# ================================================================
# 5. WRITE TO DELTA LAKE (The Modern Standard)
# ================================================================
analytics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_db.analytics_tb")

print(f"✅ Successfully wrote to Delta Lake managed table: olist_db.analytics_tb")

Orders count: 99441
Customers count: 99441
✅ Successfully wrote to Delta Lake managed table: olist_db.analytics_tb


In [0]:
# Read the Delta table back using the Metastore
delta_df = spark.table("olist_db.analytics_tb")

display(delta_df.select("order_id", "customer_state", "delivery_time_days").limit(10))

order_id,customer_state,delivery_time_days
e481f51cbdc54678b7cc49136f2d6af7,SP,8
53cdb2fc8bc7dce0b6741e2150273451,BA,14
47770eb9100c2d0c44946d9cf07ec65d,GO,9
949d5b44dbf5de918fe9c16f97b45f8a,RN,14
ad21c59c0840e6cb83a9ceb5573f8159,SP,3
a4591c265e18cb1dcee52889e2d8acc3,PR,17
136cce7faa42fdb2cefd53fdc79a6098,RS,-1
6514b8ad8028c9f2cc2374ded245783f,RJ,10
76c6e866289321a7c93b82b54852dc33,RS,10
e69bfb5eb88e0ed6a785585b27e16dbf,SP,18
